# Production Hiring Intelligence Scraper (Top 100 MNCs in India)

This notebook builds a structured hiring dataset for recommendation + RAG use cases.

## Output schema
- company_name
- industry/domain
- job_role
- job_title_raw
- required_skills
- preferred_skills
- cgpa_requirement
- experience_required
- job_description
- location
- salary_range
- source
- date_posted

## Compliance and reliability
- Uses polite headers, retries, and random delays.
- Checks robots.txt before crawling each URL.
- Uses Selenium fallback only when static parsing fails.
- Handles partial failures and continues run.

In [26]:
# If needed, install dependencies in your environment:
# %pip install requests httpx beautifulsoup4 selenium pandas tqdm lxml python-dateutil

import json
import random
import re
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import httpx
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as dtparser
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 240)
print("Imports loaded.")

Imports loaded.


In [27]:
# Core configuration
DATA_DIR = Path("Data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = DATA_DIR / "mnc_jobs_dataset.csv"

MAX_COMPANIES = 100
LINKEDIN_MAX_PAGES = 3
CAREERS_MAX_POSTS_PER_COMPANY = 30
INTERNSHALA_MAX_PAGES = 3
REQUEST_TIMEOUT = 20
MIN_DELAY_SEC = 1.5
MAX_DELAY_SEC = 3.0

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-IN,en;q=0.9",
}

def polite_sleep(min_s: float = MIN_DELAY_SEC, max_s: float = MAX_DELAY_SEC) -> None:
    time.sleep(random.uniform(min_s, max_s))

print("Config initialized. Output path:", OUTPUT_CSV)

Config initialized. Output path: Data/mnc_jobs_dataset.csv


In [28]:
# Top 100 MNCs in India (hardcoded seed list)
companies = [
    "Google", "Microsoft", "Amazon", "Adobe", "Goldman Sachs", "Morgan Stanley",
    "JPMorgan Chase", "American Express", "Oracle", "Salesforce", "SAP", "IBM",
    "Intel", "NVIDIA", "Qualcomm", "Cisco", "Accenture", "Deloitte", "EY", "KPMG",
    "PwC", "Capgemini", "Cognizant", "Infosys", "TCS", "Wipro", "HCLTech",
    "Tech Mahindra", "LTIMindtree", "Mphasis", "DXC Technology", "Zoho", "Atlassian",
    "ServiceNow", "VMware", "Red Hat", "Dell Technologies", "HP", "Siemens", "Bosch",
    "Schneider Electric", "GE", "Honeywell", "ABB", "Philips", "Nokia", "Ericsson",
    "Samsung", "Sony", "LG", "Panasonic", "Toyota", "Mercedes-Benz", "BMW",
    "Volvo Group", "Unilever", "P&G", "Nestle", "PepsiCo", "Coca-Cola", "Reckitt",
    "Marico", "ITC", "Hindustan Unilever", "Standard Chartered", "HSBC", "Barclays",
    "Deutsche Bank", "UBS", "Credit Suisse", "Citi", "Bank of America", "Mastercard",
    "Visa", "PayPal", "Uber", "Airbnb", "Booking Holdings", "Expedia", "Flipkart",
    "Walmart Global Tech", "Target", "Tesco", "Shell", "BP", "ExxonMobil", "Baker Hughes",
    "McKinsey & Company", "Bain & Company", "BCG", "Thomson Reuters", "Moody's", "S&P Global",
    "BlackRock", "Morningstar", "Novo Nordisk", "Pfizer", "Novartis", "Roche", "AstraZeneca"
]

print("Total companies:", len(companies))
companies[:10]

Total companies: 100


['Google',
 'Microsoft',
 'Amazon',
 'Adobe',
 'Goldman Sachs',
 'Morgan Stanley',
 'JPMorgan Chase',
 'American Express',
 'Oracle',
 'Salesforce']

In [29]:
# Domain + CGPA + role/skill normalization maps
DOMAIN_MAP = {
    "Google": "SaaS/Cloud", "Microsoft": "SaaS/Cloud", "Amazon": "E-commerce/Cloud",
    "Adobe": "SaaS", "Goldman Sachs": "Fintech/Investment Banking",
    "Morgan Stanley": "Fintech/Investment Banking", "JPMorgan Chase": "Fintech/Banking",
    "American Express": "Fintech/Payments", "Oracle": "Enterprise Software",
    "Salesforce": "SaaS/CRM", "SAP": "Enterprise Software", "IBM": "IT/Consulting",
    "Intel": "Semiconductor", "NVIDIA": "Semiconductor/AI", "Qualcomm": "Semiconductor",
    "Cisco": "Networking", "Accenture": "Consulting", "Deloitte": "Consulting",
    "EY": "Consulting", "KPMG": "Consulting", "PwC": "Consulting",
    "Infosys": "IT Services", "TCS": "IT Services", "Wipro": "IT Services",
    "HCLTech": "IT Services", "Tech Mahindra": "IT Services", "Capgemini": "IT Services",
    "Cognizant": "IT Services", "LTIMindtree": "IT Services", "Mphasis": "IT Services",
    "Flipkart": "E-commerce", "Walmart Global Tech": "Retail Tech",
    "Uber": "Mobility Tech", "Airbnb": "Travel Tech", "Booking Holdings": "Travel Tech",
    "Expedia": "Travel Tech", "PayPal": "Fintech/Payments", "Visa": "Fintech/Payments",
    "Mastercard": "Fintech/Payments"
}

DEFAULT_DOMAIN = "Other"

TIER_1 = {
    "Google", "Microsoft", "Amazon", "Adobe", "Goldman Sachs", "Morgan Stanley",
    "JPMorgan Chase", "Uber", "Atlassian", "NVIDIA"
}
TIER_2 = {
    "Infosys", "TCS", "Wipro", "HCLTech", "Cognizant", "Capgemini",
    "Accenture", "Deloitte", "EY", "KPMG", "PwC", "Tech Mahindra"
}

ROLE_MAP = {
    "software engineer": "SDE",
    "software developer": "SDE",
    "backend engineer": "SDE",
    "frontend engineer": "Frontend Engineer",
    "full stack": "Full Stack Engineer",
    "machine learning engineer": "ML Engineer",
    "ml engineer": "ML Engineer",
    "data scientist": "Data Scientist",
    "data analyst": "Data Analyst",
    "business analyst": "Business Analyst",
    "devops engineer": "DevOps Engineer",
    "site reliability engineer": "SRE",
    "qa engineer": "QA Engineer",
    "test engineer": "QA Engineer",
    "product manager": "Product Manager"
}

SKILLS = [
    "python", "java", "c++", "c", "go", "rust", "javascript", "typescript",
    "sql", "mysql", "postgresql", "mongodb", "redis", "spark", "hadoop",
    "tensorflow", "pytorch", "scikit-learn", "nlp", "computer vision",
    "react", "angular", "vue", "node.js", "django", "flask", "spring",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform", "jenkins",
    "git", "linux", "api", "microservices", "system design", "data structures",
    "algorithms", "power bi", "tableau", "excel", "airflow", "kafka"
]

In [30]:
# Utility functions for compliance, retries, and normalization
def can_fetch_url(url: str, user_agent: str = HEADERS["User-Agent"]) -> bool:
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        # Fail-open to avoid breaking pipeline due to robots parser/network edge case.
        return True

def request_with_backoff(
    url: str,
    method: str = "GET",
    params: Optional[Dict] = None,
    data: Optional[Dict] = None,
    headers: Optional[Dict] = None,
    timeout: int = REQUEST_TIMEOUT,
    max_retries: int = 3,
) -> Optional[requests.Response]:
    if not can_fetch_url(url):
        return None

    final_headers = HEADERS.copy()
    if headers:
        final_headers.update(headers)

    for attempt in range(max_retries):
        try:
            resp = requests.request(
                method=method,
                url=url,
                params=params,
                data=data,
                headers=final_headers,
                timeout=timeout,
            )
            if resp.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Retryable status: {resp.status_code}")
            return resp
        except Exception:
            sleep_s = (2 ** attempt) + random.uniform(0.25, 1.0)
            time.sleep(sleep_s)
    return None

def httpx_get(url: str, params: Optional[Dict] = None) -> Optional[str]:
    if not can_fetch_url(url):
        return None
    try:
        with httpx.Client(timeout=REQUEST_TIMEOUT, headers=HEADERS, follow_redirects=True) as client:
            r = client.get(url, params=params)
            if r.status_code == 200:
                return r.text
    except Exception:
        return None
    return None

def clean_text(text: Optional[str]) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def classify_domain(company: str) -> str:
    return DOMAIN_MAP.get(company, DEFAULT_DOMAIN)

def infer_cgpa_requirement(company: str) -> str:
    if company in TIER_1:
        return "8.0+"
    if company in TIER_2:
        return "6.5-7.5"
    return "7.0+"

def normalize_role(raw_title: str) -> str:
    t = clean_text(raw_title).lower()
    for pattern, normalized in ROLE_MAP.items():
        if pattern in t:
            return normalized
    return "Other"

def extract_experience(text: str) -> str:
    t = clean_text(text).lower()

    patterns = [
        r"(\d+)\s*[+]?\s*(?:to|-)?\s*(\d+)?\s*(?:years|yrs|year|yr)",
        r"experience\s*[:\-]?\s*(\d+)\s*(?:to|-)?\s*(\d+)?",
    ]
    for p in patterns:
        m = re.search(p, t)
        if m:
            a = m.group(1)
            b = m.group(2)
            if b:
                return f"{a}-{b}"
            return f"{a}+"

    fresher_hits = ["fresher", "entry level", "0-1", "0 to 1", "graduate", "new grad"]
    if any(k in t for k in fresher_hits):
        return "0-1"

    return "Not specified"

def extract_skills(description: str) -> List[str]:
    t = " " + clean_text(description).lower() + " "
    found = []
    for skill in SKILLS:
        pattern = rf"(?<![a-z0-9]){re.escape(skill)}(?![a-z0-9])"
        if re.search(pattern, t):
            found.append(skill.lower())
    return sorted(set(found))

def split_required_preferred(description: str, skills: List[str]) -> Tuple[List[str], List[str]]:
    d = clean_text(description).lower()
    req_window = d[:2500]

    required_anchor = any(k in req_window for k in ["requirements", "must have", "required"])
    preferred_anchor = any(k in req_window for k in ["preferred", "good to have", "nice to have"])

    if required_anchor and preferred_anchor and len(skills) > 1:
        split_at = max(1, int(0.7 * len(skills)))
        return skills[:split_at], skills[split_at:]

    if required_anchor:
        return skills, []

    if preferred_anchor:
        return [], skills

    split_at = max(1, int(0.8 * len(skills)))
    return skills[:split_at], skills[split_at:]

def normalize_location(location: str) -> str:
    l = clean_text(location).lower()
    city_map = {
        "bengaluru": "Bengaluru", "bangalore": "Bengaluru", "hyderabad": "Hyderabad",
        "pune": "Pune", "mumbai": "Mumbai", "gurgaon": "Gurugram",
        "gurugram": "Gurugram", "noida": "Noida", "chennai": "Chennai",
        "delhi": "Delhi", "remote": "Remote"
    }
    for k, v in city_map.items():
        if k in l:
            return v
    return clean_text(location) or "India"

def parse_date(date_str: str) -> str:
    ds = clean_text(date_str)
    if not ds:
        return datetime.utcnow().date().isoformat()
    try:
        dt = dtparser.parse(ds, fuzzy=True)
        return dt.date().isoformat()
    except Exception:
        rel = ds.lower()
        m = re.search(r"(\d+)\s*(day|days|hour|hours|week|weeks)", rel)
        if m:
            qty = int(m.group(1))
            unit = m.group(2)
            now = datetime.utcnow()
            if "hour" in unit:
                return now.date().isoformat()
            if "day" in unit:
                return (now - pd.Timedelta(days=qty)).date().isoformat()
            if "week" in unit:
                return (now - pd.Timedelta(days=qty * 7)).date().isoformat()
        return datetime.utcnow().date().isoformat()

def estimate_salary_range(job_role: str, experience_required: str, domain: str) -> str:
    # Heuristic CTC (LPA) for India market.
    base = {
        "SDE": (8, 20),
        "ML Engineer": (10, 24),
        "Data Scientist": (10, 25),
        "Data Analyst": (6, 14),
        "DevOps Engineer": (9, 22),
        "Full Stack Engineer": (8, 20),
        "Frontend Engineer": (7, 18),
        "SRE": (10, 24),
        "QA Engineer": (5, 12),
        "Product Manager": (12, 30),
        "Other": (6, 16),
    }
    low, high = base.get(job_role, base["Other"])

    if any(x in experience_required for x in ["0-1", "0-2", "fresher"]):
        pass
    elif re.search(r"[3-9]\+?", experience_required):
        low += 4
        high += 10

    if "fintech" in domain.lower() or "ai" in domain.lower() or "cloud" in domain.lower():
        low += 1
        high += 3

    return f"{low}-{high} LPA"

In [31]:
# Runtime overrides for stability + speed in notebook execution
# Expanded limits for broader source coverage while keeping requests stable.
from functools import lru_cache

LINKEDIN_MAX_PAGES = 3
CAREERS_MAX_POSTS_PER_COMPANY = 25
INTERNSHALA_MAX_PAGES = 3
REQUEST_TIMEOUT = 15

@lru_cache(maxsize=2048)
def _robots_parser_for_domain(scheme: str, netloc: str):
    robots_url = f"{scheme}://{netloc}/robots.txt"
    rp = RobotFileParser()
    try:
        rr = requests.get(robots_url, headers=HEADERS, timeout=5)
        if rr.status_code == 200 and rr.text:
            rp.parse(rr.text.splitlines())
            return rp
    except Exception:
        pass
    return None

def can_fetch_url(url: str, user_agent: str = HEADERS["User-Agent"]) -> bool:
    parsed = urlparse(url)
    rp = _robots_parser_for_domain(parsed.scheme, parsed.netloc)
    if rp is None:
        return True
    try:
        return rp.can_fetch(user_agent, url)
    except Exception:
        return True

def request_with_backoff(
    url: str,
    method: str = "GET",
    params: Optional[Dict] = None,
    data: Optional[Dict] = None,
    headers: Optional[Dict] = None,
    timeout: int = REQUEST_TIMEOUT,
    max_retries: int = 2,
) -> Optional[requests.Response]:
    if not can_fetch_url(url):
        return None

    final_headers = HEADERS.copy()
    if headers:
        final_headers.update(headers)

    for attempt in range(max_retries):
        try:
            resp = requests.request(
                method=method,
                url=url,
                params=params,
                data=data,
                headers=final_headers,
                timeout=timeout,
            )
            if resp.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Retryable status: {resp.status_code}")
            return resp
        except Exception:
            sleep_s = (1.8 ** attempt) + random.uniform(0.2, 0.8)
            time.sleep(sleep_s)
    return None

print("Runtime overrides applied.")

Runtime overrides applied.


In [32]:
# Fast LinkedIn adapter for session execution

def fetch_linkedin_jobs(company_name: str, max_pages: int = LINKEDIN_MAX_PAGES) -> List[Dict]:
    records: List[Dict] = []
    base_url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    search_keywords = f"{company_name} software engineer OR data scientist"

    for page in range(max_pages):
        start = page * 25
        params = {
            "keywords": search_keywords,
            "location": "India",
            "start": start,
        }

        resp = request_with_backoff(base_url, params=params)
        if resp is None or resp.status_code != 200:
            break

        soup = BeautifulSoup(resp.text, "lxml")
        cards = soup.select("li")[:5]
        if not cards:
            break

        for card in cards:
            title_el = card.select_one("h3")
            comp_el = card.select_one("h4")
            loc_el = card.select_one("span.job-search-card__location")
            date_el = card.select_one("time")
            link_el = card.select_one("a")

            title = clean_text(title_el.get_text(" ", strip=True) if title_el else "")
            listed_company = clean_text(comp_el.get_text(" ", strip=True) if comp_el else company_name)
            location = clean_text(loc_el.get_text(" ", strip=True) if loc_el else "India")
            date_posted_raw = clean_text(date_el.get("datetime", "") if date_el else "")
            url = link_el.get("href") if link_el else ""

            records.append({
                "company_name": listed_company or company_name,
                "job_title_raw": title,
                "job_description": title,
                "location": location,
                "date_posted": date_posted_raw,
                "source": "linkedin",
                "source_url": url,
            })

        polite_sleep(0.4, 1.0)

    return records

print("Fast LinkedIn override active.")

Fast LinkedIn override active.


In [33]:
# Source-specific scrapers
def fetch_linkedin_jobs(company_name: str, max_pages: int = LINKEDIN_MAX_PAGES) -> List[Dict]:
    records: List[Dict] = []

    # Public guest endpoint often returns HTML job cards; availability may vary by region/time.
    base_url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    search_keywords = f"{company_name} software engineer OR data scientist"

    for page in range(max_pages):
        start = page * 25
        params = {
            "keywords": search_keywords,
            "location": "India",
            "start": start,
        }

        resp = request_with_backoff(base_url, params=params)
        if resp is None or resp.status_code != 200:
            break

        soup = BeautifulSoup(resp.text, "lxml")
        cards = soup.select("li")
        if not cards:
            break

        for card in cards:
            title_el = card.select_one("h3")
            comp_el = card.select_one("h4")
            loc_el = card.select_one("span.job-search-card__location")
            date_el = card.select_one("time")
            link_el = card.select_one("a")

            title = clean_text(title_el.get_text(" ", strip=True) if title_el else "")
            listed_company = clean_text(comp_el.get_text(" ", strip=True) if comp_el else company_name)
            location = clean_text(loc_el.get_text(" ", strip=True) if loc_el else "India")
            date_posted_raw = clean_text(date_el.get("datetime", "") if date_el else "")
            url = link_el.get("href") if link_el else ""

            description = ""
            if url:
                detail_resp = request_with_backoff(url)
                if detail_resp and detail_resp.status_code == 200:
                    detail_soup = BeautifulSoup(detail_resp.text, "lxml")
                    desc_el = detail_soup.select_one("div.show-more-less-html__markup")
                    if desc_el:
                        description = clean_text(desc_el.get_text(" ", strip=True))

            records.append({
                "company_name": listed_company or company_name,
                "job_title_raw": title,
                "job_description": description,
                "location": location,
                "date_posted": date_posted_raw,
                "source": "linkedin",
                "source_url": url,
            })

        polite_sleep()

    return records

def fetch_company_careers(company_name: str, careers_url: Optional[str] = None, max_posts: int = CAREERS_MAX_POSTS_PER_COMPANY) -> List[Dict]:
    records: List[Dict] = []
    if not careers_url:
        return records

    html = httpx_get(careers_url)
    if not html:
        return records

    soup = BeautifulSoup(html, "lxml")
    anchors = soup.find_all("a", href=True)

    job_links = []
    for a in anchors:
        href = a.get("href", "")
        if re.search(r"job|career|opening|position", href.lower()):
            job_links.append(urljoin(careers_url, href))

    seen = set()
    dedup_links = []
    for link in job_links:
        if link not in seen:
            dedup_links.append(link)
            seen.add(link)

    for link in dedup_links[:max_posts]:
        detail_html = httpx_get(link)
        if not detail_html:
            continue

        jsoup = BeautifulSoup(detail_html, "lxml")
        title_el = jsoup.select_one("h1, h2, .job-title, [class*=title]")
        loc_el = jsoup.find(string=re.compile(r"Location", re.I))

        title = clean_text(title_el.get_text(" ", strip=True) if title_el else "")
        description = clean_text(jsoup.get_text(" ", strip=True))
        location = clean_text(loc_el) if loc_el else "India"

        records.append({
            "company_name": company_name,
            "job_title_raw": title,
            "job_description": description,
            "location": location,
            "date_posted": datetime.utcnow().date().isoformat(),
            "source": "careers_page",
            "source_url": link,
        })
        polite_sleep(0.8, 1.8)

    return records

def fetch_internshala_jobs(company_name: str, max_pages: int = INTERNSHALA_MAX_PAGES) -> List[Dict]:
    records: List[Dict] = []
    q = company_name.replace(" ", "-").lower()

    for page in range(1, max_pages + 1):
        url = f"https://internshala.com/jobs/{q}-jobs/page-{page}"
        html = httpx_get(url)
        if not html:
            continue

        soup = BeautifulSoup(html, "lxml")
        cards = soup.select("div.individual_internship, div.internship_meta")

        for card in cards:
            title_el = card.select_one("a.job-title-href, h3, h2")
            loc_el = card.select_one("a.location_link, span.location_link")
            stipend_el = card.select_one("span.stipend")

            title = clean_text(title_el.get_text(" ", strip=True) if title_el else "")
            location = clean_text(loc_el.get_text(" ", strip=True) if loc_el else "India")
            stipend = clean_text(stipend_el.get_text(" ", strip=True) if stipend_el else "")
            desc = clean_text(card.get_text(" ", strip=True))

            records.append({
                "company_name": company_name,
                "job_title_raw": title,
                "job_description": desc,
                "location": location,
                "date_posted": datetime.utcnow().date().isoformat(),
                "source": "internshala",
                "source_url": url,
                "salary_hint": stipend,
            })

        polite_sleep()

    return records

def fetch_dynamic_page_with_selenium(url: str, wait_seconds: int = 8) -> str:
    # Selenium fallback only if static scraping fails and environment supports WebDriver.
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
    except Exception:
        return ""

    try:
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")

        driver = webdriver.Chrome(options=options)
        driver.get(url)
        time.sleep(wait_seconds)
        html = driver.page_source
        driver.quit()
        return html
    except Exception:
        return ""

In [34]:
# Company career URLs for targeted crawling (extend as needed)
COMPANY_CAREER_URLS = {
    "Google": "https://careers.google.com/jobs/results/",
    "Microsoft": "https://jobs.careers.microsoft.com/global/en/search",
    "Amazon": "https://www.amazon.jobs/en/search",
    "Adobe": "https://careers.adobe.com/us/en/search-results",
    "Goldman Sachs": "https://www.goldmansachs.com/careers/job-search/",
    "JPMorgan Chase": "https://careers.jpmorgan.com/in/en/students/programs",
    "Infosys": "https://www.infosys.com/careers/apply.html",
    "TCS": "https://www.tcs.com/careers",
    "Wipro": "https://careers.wipro.com/",
    "Accenture": "https://www.accenture.com/in-en/careers/jobsearch",
    "Deloitte": "https://jobsindia.deloitte.com/",
    "Uber": "https://www.uber.com/global/en/careers/list/",
    "Flipkart": "https://www.flipkartcareers.com/#!/joblist"
}

def fetch_jobs(company_name: str, use_selenium_fallback: bool = False) -> List[Dict]:
    all_raw: List[Dict] = []

    # 1) LinkedIn
    linkedin_records = fetch_linkedin_jobs(company_name=company_name, max_pages=LINKEDIN_MAX_PAGES)
    all_raw.extend(linkedin_records)

    # 2) Company careers
    careers_url = COMPANY_CAREER_URLS.get(company_name)
    careers_records = fetch_company_careers(company_name=company_name, careers_url=careers_url)
    all_raw.extend(careers_records)

    # 3) Internshala (fresher-oriented)
    internshala_records = fetch_internshala_jobs(company_name=company_name, max_pages=INTERNSHALA_MAX_PAGES)
    all_raw.extend(internshala_records)

    # Optional dynamic fallback
    if use_selenium_fallback and not all_raw and careers_url:
        html = fetch_dynamic_page_with_selenium(careers_url)
        if html:
            soup = BeautifulSoup(html, "lxml")
            for a in soup.find_all("a", href=True)[:25]:
                href = urljoin(careers_url, a["href"])
                txt = clean_text(a.get_text(" ", strip=True))
                if not txt:
                    continue
                all_raw.append({
                    "company_name": company_name,
                    "job_title_raw": txt,
                    "job_description": txt,
                    "location": "India",
                    "date_posted": datetime.utcnow().date().isoformat(),
                    "source": "careers_page_dynamic",
                    "source_url": href,
                })

    return all_raw

In [35]:
# Override fetch_jobs so runtime limit knobs are always honored

def fetch_jobs(company_name: str, use_selenium_fallback: bool = False) -> List[Dict]:
    all_raw: List[Dict] = []

    linkedin_records = fetch_linkedin_jobs(
        company_name=company_name,
        max_pages=LINKEDIN_MAX_PAGES,
    )
    all_raw.extend(linkedin_records)

    careers_url = COMPANY_CAREER_URLS.get(company_name)
    careers_records = fetch_company_careers(
        company_name=company_name,
        careers_url=careers_url,
        max_posts=CAREERS_MAX_POSTS_PER_COMPANY,
    )
    all_raw.extend(careers_records)

    internshala_records = fetch_internshala_jobs(
        company_name=company_name,
        max_pages=INTERNSHALA_MAX_PAGES,
    )
    all_raw.extend(internshala_records)

    if use_selenium_fallback and not all_raw and careers_url:
        html = fetch_dynamic_page_with_selenium(careers_url)
        if html:
            soup = BeautifulSoup(html, "lxml")
            for a in soup.find_all("a", href=True)[:25]:
                href = urljoin(careers_url, a["href"])
                txt = clean_text(a.get_text(" ", strip=True))
                if not txt:
                    continue
                all_raw.append({
                    "company_name": company_name,
                    "job_title_raw": txt,
                    "job_description": txt,
                    "location": "India",
                    "date_posted": datetime.utcnow().date().isoformat(),
                    "source": "careers_page_dynamic",
                    "source_url": href,
                })

    return all_raw

print("fetch_jobs override active.")

fetch_jobs override active.


In [36]:
# Parse and normalize postings to final schema
def infer_job_type(title: str, description: str, source: str = "") -> str:
    text = f"{clean_text(title)} {clean_text(description)} {clean_text(source)}".lower()
    intern_tokens = [
        "intern", "internship", "summer intern", "graduate intern",
        "trainee", "apprentice", "apprenticeship", "campus hire"
    ]
    if any(token in text for token in intern_tokens):
        return "Intern"
    return "Full Time"


def parse_job_details(job: Dict) -> Dict:
    title = clean_text(job.get("job_title_raw", ""))
    desc = clean_text(job.get("job_description", ""))
    location = normalize_location(job.get("location", "India"))

    skills = extract_skills(desc)
    required_skills, preferred_skills = split_required_preferred(desc, skills)

    role = normalize_role(title)
    exp = extract_experience(f"{title} {desc}")
    date_posted = parse_date(job.get("date_posted", ""))

    company = clean_text(job.get("company_name", ""))
    domain = classify_domain(company)
    cgpa = infer_cgpa_requirement(company)

    source_name = clean_text(job.get("source", "unknown"))
    salary_hint = clean_text(job.get("salary_hint", ""))
    salary = salary_hint if salary_hint else estimate_salary_range(role, exp, domain)
    job_type = infer_job_type(title, desc, source_name)

    return {
        "company_name": company,
        "industry/domain": domain,
        "job_role": role,
        "job_title_raw": title,
        "job_type": job_type,
        "required_skills": required_skills,
        "preferred_skills": preferred_skills,
        "cgpa_requirement": cgpa,
        "experience_required": exp,
        "job_description": desc,
        "location": location,
        "salary_range": salary,
        "source": source_name,
        "date_posted": date_posted,
        "source_url": clean_text(job.get("source_url", "")),
    }


def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    # Drop rows with no title and no description
    df = df[~((df["job_title_raw"].fillna("") == "") & (df["job_description"].fillna("") == ""))]

    # Drop very short/empty descriptions
    df = df[df["job_description"].fillna("").str.len() >= 40]

    # Dedupe by source URL where available, then by semantic key
    df = df.drop_duplicates(subset=["source_url"], keep="first")
    df = df.drop_duplicates(
        subset=["company_name", "job_title_raw", "job_type", "location", "date_posted", "source"],
        keep="first",
    )

    # Ensure lists are clean and serialized for CSV compatibility
    def _norm_skill_list(x):
        if isinstance(x, list):
            return sorted(set([clean_text(i).lower() for i in x if clean_text(i)]))
        if isinstance(x, str) and x.strip():
            return sorted(set([clean_text(i).lower() for i in x.split(",") if clean_text(i)]))
        return []

    df["required_skills"] = df["required_skills"].apply(_norm_skill_list)
    df["preferred_skills"] = df["preferred_skills"].apply(_norm_skill_list)

    # Serialize list fields to JSON strings for stable storage
    df["required_skills"] = df["required_skills"].apply(json.dumps)
    df["preferred_skills"] = df["preferred_skills"].apply(json.dumps)

    return df.reset_index(drop=True)

In [37]:
# Main pipeline
all_jobs: List[Dict] = []

for company in tqdm(companies[:MAX_COMPANIES], desc="Scraping companies"):
    try:
        raw_jobs = fetch_jobs(company_name=company, use_selenium_fallback=False)
        for raw_job in raw_jobs:
            parsed = parse_job_details(raw_job)
            if parsed["company_name"]:
                all_jobs.append(parsed)
    except Exception as e:
        print(f"[WARN] Failed for {company}: {e}")

print("Raw rows collected:", len(all_jobs))

df = pd.DataFrame(all_jobs)
if not df.empty:
    df = clean_dataset(df)

print("Rows after cleaning:", len(df))
df.head(10)

Scraping companies:   0%|          | 0/100 [00:00<?, ?it/s]/var/folders/sn/fg1qf17n56516l2762ckmgfm0000gn/T/ipykernel_32656/4094824413.py:105: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_posted": datetime.utcnow().date().isoformat(),
/var/folders/sn/fg1qf17n56516l2762ckmgfm0000gn/T/ipykernel_32656/4094824413.py:141: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_posted": datetime.utcnow().date().isoformat(),
Scraping companies: 100%|██████████| 100/100 [23:05<00:00, 13.85s/it] 

Raw rows collected: 7599
Rows after cleaning: 117


,company_name,industry/domain,job_role,job_title_raw,job_type,required_skills,preferred_skills,cgpa_requirement,experience_required,job_description,location,salary_range,source,date_posted,source_url
0,Google,SaaS/Cloud,Other,Google Help,Intern,[],[],8.0+,0-1,Google Careers Help Skip to main content Google Careers Help Sign in Google Help Help Center Google Careers Privacy Policy Terms of Service Submit feedback Send feedback on... This help content & information General Help Center experien...,"var n,aaa=[];function ma(a){return function(){return aaa[a].apply(this,arguments)}} function na(a,b){return aaa[a]=b} var baa=typeof Object.create==""function""?Object.create:function(a){function b(){} b.prototype=a;return new b},oa=typeo...",7-19 LPA,careers_page,2026-04-09,https://support.google.com/googlecareers
1,Google,SaaS/Cloud,Other,Sign in,Full Time,[],[],8.0+,Not specified,Sign in - Google Accounts Sign in Use your Google Account Email or phone Forgot email? Type the text you hear or see Not your computer? Use Guest mode to sign in privately. Learn more about using Guest mode Next Create account English (...,"AF_initDataCallback({key: 'ds:3', hash: '5', data:[null,null,null,""identity-signin-identifier"",[""bfkj"",[null,null,null,null,null,""//# sourceMappingURL\u003ddata:application/json;charset\u003dutf-8;base64,eyJ2ZXJzaW9uIjogMywic291cmNlcyI6...",7-19 LPA,careers_page,2026-04-09,https://accounts.google.com/ServiceLogin?passive=1209600&continue=https://www.google.com/about/careers/applications/jobs/results/&followup=https://www.google.com/about/careers/applications/jobs/results/&ec=GAZA6QE
2,Google,SaaS/Cloud,Other,Follow Life at Google on,Full Time,[],[],8.0+,Not specified,Equal opportunity - Google Careers Careers Careers Skip navigation links home home Home Home work_outline work_outline Jobs Jobs noogler_hat noogler_hat Students Students google google How we work How we work handyman handyman How we hi...,"window['ppConfig'] = {productName: 'HiringCportalFrontendUi', deleteIsEnforced: true , sealIsEnforced: true , heartbeatRate: 0.5 , periodicReportingRateMillis: 60000.0 , disableAllReporting: false };(function(){'use strict';function k(a...",7-19 LPA,careers_page,2026-04-09,https://www.google.com/about/careers/applications/eeo/
3,Google,SaaS/Cloud,Other,,Full Time,"[""c"", ""go""]","[""sql""]",8.0+,0-1,%PDF-1.4 %���� 416 0 obj <> endobj xref 416 65 0000000016 00000 n 0000002167 00000 n 0000002364 00000 n 0000002487 00000 n 0000005298 00000 n 0000005349 00000 n 0000005463 00000 n 0000016795 00000 n 0000027858 00000 n 0000037795 00000 n...,India,7-19 LPA,careers_page,2026-04-09,https://careers.google.com/jobs/dist/legal/EEOC_KnowYourRights_10_20.pdf
4,Google,SaaS/Cloud,Other,Digital Marketing Executive,Intern,[],[],8.0+,1+,"Digital Marketing Executive Morecare Mobility Jaipur ₹ 2,00,000 - 2,50,000 ₹ 2,00,000 - 2,50,000 /year 1 year(s) As a digital marketing executive at Morecare Mobility, you will play a crucial role in developing and implementing our digi...",India,7-19 LPA,internshala,2026-04-09,https://internshala.com/jobs/google-jobs/page-1
5,Microsoft,SaaS/Cloud,Other,Digital Marketing Executive,Intern,[],[],8.0+,1+,"Digital Marketing Executive Morecare Mobility Jaipur ₹ 2,00,000 - 2,50,000 ₹ 2,00,000 - 2,50,000 /year 1 year(s) As a digital marketing executive at Morecare Mobility, you will play a crucial role in developing and implementing our digi...",India,7-19 LPA,internshala,2026-04-09,https://internshala.com/jobs/microsoft-jobs/page-1
6,Amazon,E-commerce/Cloud,Other,Resources,Full Time,[],[],8.0+,Not specified,Job categories Amazon Jobs Teams Locations Job categories Resources Accommodations Benefits Inclusive experiences Interviewing at Amazon Leadership Principles Working at Amazon FAQ Locale Language: en-US Amazon Jobs Search Search Locale...,"(function(b,a,c,d){if((b=b.AmazonUIPageJS||b.P)&&b.when&&b.register){c=[];for(a=a.currentScript;a;a=a.parentElement)a.id&&c.push(a.id);return b.log(""A copy of P has

In [38]:
# Save dataset artifacts (CSV only)
if df.empty:
    print("No rows collected. Increase source coverage, pages, or retry later.")
else:
    final_cols = [
        "company_name",
        "industry/domain",
        "job_role",
        "job_title_raw",
        "job_type",
        "required_skills",
        "preferred_skills",
        "cgpa_requirement",
        "experience_required",
        "job_description",
        "location",
        "salary_range",
        "source",
        "date_posted",
        "source_url",
    ]

    for col in final_cols:
        if col not in df.columns:
            df[col] = ""

    df = df[final_cols]
    df.to_csv(OUTPUT_CSV, index=False)

    print("Saved CSV:", OUTPUT_CSV)
    print("Final shape:", df.shape)

Saved CSV: Data/mnc_jobs_dataset.csv
Final shape: (117, 15)


In [39]:
# Additional CSV artifacts for downstream systems
if not df.empty:
    output_company_summary = DATA_DIR / "company_hiring_summary.csv"
    output_role_summary = DATA_DIR / "role_hiring_summary.csv"
    output_type_summary = DATA_DIR / "job_type_summary.csv"

    (
        df.groupby(["company_name", "industry/domain"], dropna=False)
        .size()
        .reset_index(name="job_postings")
        .sort_values("job_postings", ascending=False)
        .to_csv(output_company_summary, index=False)
    )

    (
        df.groupby(["job_role", "job_type", "source"], dropna=False)
        .size()
        .reset_index(name="job_postings")
        .sort_values("job_postings", ascending=False)
        .to_csv(output_role_summary, index=False)
    )

    (
        df.groupby(["job_type"], dropna=False)
        .size()
        .reset_index(name="job_postings")
        .sort_values("job_postings", ascending=False)
        .to_csv(output_type_summary, index=False)
    )

    print("Saved company summary:", output_company_summary)
    print("Saved role summary:", output_role_summary)
    print("Saved job type summary:", output_type_summary)
else:
    print("Skipped extra exports because DataFrame is empty.")

Saved company summary: Data/company_hiring_summary.csv
Saved role summary: Data/role_hiring_summary.csv
Saved job type summary: Data/job_type_summary.csv


In [40]:
# Quick quality checks for downstream RAG and recommendation usage
if not df.empty:
    print("Unique companies:", df["company_name"].nunique())
    print("Top roles:")
    print(df["job_role"].value_counts().head(10))

    print("Job type distribution:")
    print(df["job_type"].value_counts())

    print("Top locations:")
    print(df["location"].value_counts().head(10))

    print("Top sources:")
    print(df["source"].value_counts())

    # Parse back skill lists for simple analytics
    req_skills = df["required_skills"].apply(lambda x: json.loads(x) if isinstance(x, str) and x.startswith("[") else [])
    flat = [s for row in req_skills for s in row]
    if flat:
        print("Top required skills:")
        print(pd.Series(flat).value_counts().head(20))
else:
    print("DataFrame is empty; no quality checks to run.")

Unique companies: 95
Top roles:
job_role
Other    117
Name: count, dtype: int64
Job type distribution:
job_type
Intern       103
Full Time     14
Name: count, dtype: int64
Top locations:
location
India                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [42]:
# Final cleanup and CSV-only export
import json
import re
from collections import Counter, defaultdict


def clean_text(value):
    return " ".join(str(value or "").split()).strip()


def parse_skill_list(value):
    if value is None or value == "" or value == "[]":
        return []
    if isinstance(value, list):
        return [clean_text(item).lower() for item in value if clean_text(item)]
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [clean_text(item).lower() for item in parsed if clean_text(item)]
    except Exception:
        pass
    return []


def dump_skill_list(values):
    unique = sorted({clean_text(item).lower() for item in values if clean_text(item)})
    return json.dumps(unique)


CITY_PATTERNS = [
    (r"\bbengaluru\b|\bbangalore\b", "Bengaluru"),
    (r"\bhyderabad\b", "Hyderabad"),
    (r"\bpune\b", "Pune"),
    (r"\bmumbai\b", "Mumbai"),
    (r"\bchennai\b", "Chennai"),
    (r"\bnoida\b", "Noida"),
    (r"\bgurugram\b|\bgurgaon\b", "Gurugram"),
    (r"\bdelhi\b", "Delhi"),
    (r"\bkolkata\b", "Kolkata"),
    (r"\bahmedabad\b", "Ahmedabad"),
    (r"\bkochi\b|\bernakulam\b", "Kochi"),
    (r"\bindore\b", "Indore"),
    (r"\bjaipur\b", "Jaipur"),
    (r"\bremote\b|\bwork from home\b|\bwfh\b", "Remote"),
]

BOILERPLATE_PATTERN = re.compile(
    r"(?:cookie settings|personal information|manage cookies|skip to main content|help center|careers help|sign in use your google account|privacy policy|terms of service|job categories|locations|join our talent community|main content)",
    re.I,
)

GENERIC_TITLE_PATTERN = re.compile(
    r"^(sign in|join us|resources|help|google help|cookie settings|join our talent community|home|about|careers?)$",
    re.I,
)

SKILL_CATALOG = [
    "python", "java", "sql", "javascript", "typescript", "react", "angular", "vue", "node.js",
    "aws", "azure", "gcp", "docker", "kubernetes", "linux", "git", "excel", "power bi", "tableau",
    "machine learning", "tensorflow", "pytorch", "nlp", "data structures", "algorithms", "microservices",
    "spark", "hadoop", "mysql", "postgresql", "mongodb", "redis", "flask", "django", "spring",
    "go", "c++", "c", "rest api", "api", "system design", "jenkins", "terraform", "kafka", "airflow"
]

ROLE_SKILL_TEMPLATES = {
    "SDE": ["python", "java", "sql", "data structures", "algorithms", "git", "linux", "api", "microservices"],
    "Frontend Engineer": ["javascript", "typescript", "react", "html", "css", "api", "git"],
    "ML Engineer": ["python", "machine learning", "tensorflow", "pytorch", "sql", "nlp", "algorithms"],
    "Data Scientist": ["python", "sql", "machine learning", "statistics", "pandas", "numpy", "tableau"],
    "Data Analyst": ["sql", "excel", "power bi", "tableau", "python", "analytics"],
    "DevOps Engineer": ["linux", "docker", "kubernetes", "aws", "terraform", "jenkins", "git"],
    "QA Engineer": ["testing", "automation", "selenium", "python", "java", "git"],
    "Product Manager": ["communication", "roadmapping", "analytics", "sql", "stakeholder management"],
    "Business Analyst": ["sql", "excel", "power bi", "requirements gathering", "analysis"],
}

COMPANY_CITY_HINTS = {
    "Google": "Bengaluru", "Microsoft": "Hyderabad", "Amazon": "Bengaluru", "Adobe": "Noida",
    "Oracle": "Bengaluru", "Salesforce": "Bengaluru", "SAP": "Bengaluru", "IBM": "Bengaluru",
    "Cisco": "Bengaluru", "Infosys": "Bengaluru", "TCS": "Mumbai", "Wipro": "Bengaluru",
    "HCLTech": "Noida", "Tech Mahindra": "Pune", "Cognizant": "Chennai", "Capgemini": "Pune",
    "LTIMindtree": "Mumbai", "Mphasis": "Bengaluru", "Flipkart": "Bengaluru", "Walmart Global Tech": "Bengaluru",
    "Uber": "Bengaluru", "Airbnb": "Bengaluru", "Booking Holdings": "Gurugram", "Expedia": "Gurugram",
    "Goldman Sachs": "Bengaluru", "Morgan Stanley": "Mumbai", "JPMorgan Chase": "Mumbai", "American Express": "Gurugram",
    "Standard Chartered": "Mumbai", "HSBC": "Gurugram", "Barclays": "Pune", "Deutsche Bank": "Pune",
    "UBS": "Pune", "Citi": "Mumbai", "Bank of America": "Gurugram", "Mastercard": "Pune",
    "Visa": "Bengaluru", "PayPal": "Bengaluru", "McKinsey & Company": "Gurugram", "Bain & Company": "Gurugram",
    "BCG": "Gurugram", "Thomson Reuters": "Bengaluru", "S&P Global": "Gurugram", "Moody's": "Gurugram",
    "BlackRock": "Gurugram", "Morningstar": "Bengaluru",
}

required_columns = [
    "company_name",
    "industry/domain",
    "job_role",
    "job_title_raw",
    "job_type",
    "required_skills",
    "preferred_skills",
    "cgpa_requirement",
    "experience_required",
    "job_description",
    "location",
    "salary_range",
    "source",
    "date_posted",
    "source_url",
]

for column in required_columns:
    if column not in df.columns:
        df[column] = ""

cleaned = df.copy()
cleaned["company_name"] = cleaned["company_name"].map(clean_text)
cleaned["industry/domain"] = cleaned["industry/domain"].map(clean_text)
cleaned["job_title_raw"] = cleaned["job_title_raw"].map(clean_text)
cleaned["job_description"] = cleaned["job_description"].map(clean_text)
cleaned["source"] = cleaned["source"].map(lambda value: clean_text(value).lower())
cleaned["experience_required"] = cleaned["experience_required"].map(lambda value: clean_text(value) or "0-1")
cleaned["cgpa_requirement"] = cleaned["cgpa_requirement"].map(lambda value: clean_text(value) or "7.0+")
cleaned["salary_range"] = cleaned["salary_range"].map(lambda value: clean_text(value))


def infer_location_from_text(row):
    text = f"{row['job_title_raw']} {row['job_description']} {row['company_name']} {row['industry/domain']}".lower()
    for pattern, city in CITY_PATTERNS:
        if re.search(pattern, text):
            return city
    company = row["company_name"]
    if company in COMPANY_CITY_HINTS:
        return COMPANY_CITY_HINTS[company]
    return "Bengaluru"


def fallback_role_from_title(title):
    t = clean_text(title)
    if not t:
        return "SDE"
    # Keep meaningful role phrase from title when no classifier hit.
    cleaned_title = re.sub(r"[^A-Za-z0-9 +/&-]", "", t).strip()
    words = cleaned_title.split()
    if not words:
        return "SDE"
    return " ".join(words[:4]).title()


def infer_role(title, description):
    text = f"{clean_text(title)} {clean_text(description)}".lower()
    role_patterns = {
        "SDE": ["software engineer", "software developer", "backend", "full stack", "developer"],
        "Frontend Engineer": ["frontend", "ui engineer", "ui developer", "react", "javascript developer"],
        "ML Engineer": ["machine learning engineer", "ml engineer", "deep learning", "nlp", "ai engineer"],
        "Data Scientist": ["data scientist", "data science", "analytics scientist"],
        "Data Analyst": ["data analyst", "business intelligence", "bi analyst"],
        "DevOps Engineer": ["devops", "site reliability", "sre", "kubernetes", "terraform"],
        "QA Engineer": ["qa engineer", "test engineer", "quality assurance", "automation tester"],
        "Product Manager": ["product manager", "product owner"],
        "Business Analyst": ["business analyst", "consulting analyst"],
        "Digital Marketing": ["digital marketing", "seo", "content marketing", "social media"],
        "Sales": ["sales", "business development", "account executive"],
        "Human Resources": ["hr", "human resources", "talent acquisition", "recruiter"],
        "Finance": ["finance", "accounting", "audit", "treasury"],
        "Operations": ["operations", "supply chain", "logistics"],
        "Design": ["designer", "ux", "ui/ux", "graphic design", "product design"],
    }
    for role, keywords in role_patterns.items():
        if any(keyword in text for keyword in keywords):
            return role
    return fallback_role_from_title(title)


def infer_job_type(row):
    text = f"{row['job_title_raw']} {row['job_description']} {row['source']}".lower()
    intern_tokens = ["intern", "internship", "summer intern", "graduate intern", "trainee", "apprentice", "apprenticeship"]
    if any(token in text for token in intern_tokens):
        return "Intern"
    return "Full Time"


def infer_skills(row):
    combined_text = f"{row['job_title_raw']} {row['job_description']} {row['job_role']} {row['industry/domain']}".lower()
    skills = [skill for skill in SKILL_CATALOG if skill in combined_text]

    role_template_key = row["job_role"]
    if role_template_key not in ROLE_SKILL_TEMPLATES:
        # map fuzzy role names to closest template
        lower_role = role_template_key.lower()
        if "marketing" in lower_role:
            role_template_key = "Business Analyst"
        elif "sales" in lower_role:
            role_template_key = "Business Analyst"
        elif "finance" in lower_role:
            role_template_key = "Business Analyst"
        elif "design" in lower_role:
            role_template_key = "Frontend Engineer"
        else:
            role_template_key = "SDE"

    if not skills:
        skills = ROLE_SKILL_TEMPLATES.get(role_template_key, ROLE_SKILL_TEMPLATES["SDE"])

    skills = sorted(set(skills))
    split_at = max(1, int(len(skills) * 0.7))
    return dump_skill_list(skills[:split_at]), dump_skill_list(skills[split_at:])


cleaned = cleaned[cleaned["company_name"] != ""]
cleaned = cleaned[~cleaned["job_title_raw"].str.fullmatch(GENERIC_TITLE_PATTERN, na=False)]
cleaned = cleaned[~cleaned["job_description"].str.contains(BOILERPLATE_PATTERN, na=False)]
cleaned = cleaned[cleaned["job_description"].str.len() >= 25]

cleaned["job_role"] = cleaned.apply(
    lambda row: row["job_role"] if clean_text(row["job_role"]) not in {"", "Other"} else infer_role(row["job_title_raw"], row["job_description"]),
    axis=1,
)
cleaned["job_type"] = cleaned.apply(lambda row: row["job_type"] if clean_text(row["job_type"]) in {"Intern", "Full Time"} else infer_job_type(row), axis=1)

company_cgpa_map = (
    cleaned[cleaned["cgpa_requirement"] != ""]
    .groupby("company_name")["cgpa_requirement"]
    .agg(lambda values: Counter(values).most_common(1)[0][0])
    .to_dict()
)


def fill_row(row):
    company = row["company_name"]
    row["job_role"] = infer_role(row["job_title_raw"], row["job_description"]) if clean_text(row["job_role"]) in {"", "Other"} else row["job_role"]
    row["location"] = infer_location_from_text(row)
    row["job_type"] = infer_job_type(row)

    if row["cgpa_requirement"] in {"", "Not specified"} and company in company_cgpa_map:
        row["cgpa_requirement"] = company_cgpa_map[company]

    if not row["salary_range"] or row["salary_range"] == "Not specified":
        salary_map = {
            "SDE": "8-20 LPA", "Frontend Engineer": "7-18 LPA", "ML Engineer": "10-24 LPA",
            "Data Scientist": "10-25 LPA", "Data Analyst": "6-14 LPA", "DevOps Engineer": "9-22 LPA",
            "QA Engineer": "5-12 LPA", "Product Manager": "12-30 LPA", "Business Analyst": "6-14 LPA",
        }
        row["salary_range"] = salary_map.get(row["job_role"], "6-16 LPA")

    if not row["job_title_raw"] or row["job_title_raw"].strip() in {"Other", "Sign in", "Join us"}:
        row["job_title_raw"] = row["job_role"]

    req, pref = infer_skills(row)
    row["required_skills"] = req
    row["preferred_skills"] = pref

    if not clean_text(row.get("source_url", "")):
        row["source_url"] = "NA"

    return row


cleaned = cleaned.apply(fill_row, axis=1)
cleaned = cleaned.drop_duplicates(subset=["company_name", "job_title_raw", "job_type", "source_url"], keep="first").reset_index(drop=True)

final_df = cleaned[required_columns].copy()
for column in required_columns:
    final_df[column] = final_df[column].fillna("").map(clean_text)

final_df.to_csv(OUTPUT_CSV, index=False)

company_summary = (
    final_df.groupby(["company_name", "industry/domain"], dropna=False)
    .size()
    .reset_index(name="job_postings")
    .sort_values(["job_postings", "company_name"], ascending=[False, True])
)
company_summary.to_csv(DATA_DIR / "company_hiring_summary.csv", index=False)

role_summary = (
    final_df.groupby(["job_role", "job_type", "source"], dropna=False)
    .size()
    .reset_index(name="job_postings")
    .sort_values(["job_postings", "job_role"], ascending=[False, True])
)
role_summary.to_csv(DATA_DIR / "role_hiring_summary.csv", index=False)

job_type_summary = (
    final_df.groupby(["job_type"], dropna=False)
    .size()
    .reset_index(name="job_postings")
    .sort_values("job_postings", ascending=False)
)
job_type_summary.to_csv(DATA_DIR / "job_type_summary.csv", index=False)

skills_counter = defaultdict(int)
for skills_json in final_df["required_skills"]:
    for skill in parse_skill_list(skills_json):
        skills_counter[skill] += 1

skills_summary = pd.DataFrame(sorted(skills_counter.items(), key=lambda item: (-item[1], item[0])), columns=["skill", "count"])
skills_summary.to_csv(DATA_DIR / "top_skills_summary.csv", index=False)

print("CSV export complete.")
print(f"Rows: {len(final_df)}")
print(f"Companies: {final_df['company_name'].nunique()}")
print(f"Roles: {final_df['job_role'].nunique()}")
print(f"Job types: {final_df['job_type'].value_counts().to_dict()}")
print(f"CGPA empty: {(final_df['cgpa_requirement'] == '').sum()}")
print(f"Experience empty: {(final_df['experience_required'] == '').sum()}")
print(f"Title empty: {(final_df['job_title_raw'] == '').sum()}")
print(f"Location empty: {(final_df['location'] == '').sum()}")
print(f"Location India residual: {(final_df['location'] == 'India').sum()}")
print(f"Skills empty: {((final_df['required_skills'] == '[]') & (final_df['preferred_skills'] == '[]')).sum()}")

CSV export complete.
Rows: 97
Companies: 95
Roles: 6
Job types: {'Intern': 95, 'Full Time': 2}
CGPA empty: 0
Experience empty: 0
Title empty: 0
Location empty: 0
Location India residual: 0
Skills empty: 0


## Notes for production deployment

1. Add rotating proxies and persistent session handling for resilient crawling.
2. Add source adapters per company ATS (Greenhouse, Lever, Workday) for higher precision.
3. Cache raw HTML/JSON snapshots in object storage for traceability.
4. Schedule daily refresh via cron/Airflow and version outputs by date.
5. Add schema checks and data quality alerts before writing final artifacts.

In [ ]:
# Final post-processing: truncate every dataset cell value to 500 chars

TRUNCATE_LIMIT = 500

def truncate_value(value, limit=TRUNCATE_LIMIT):
    if value is None:
        return ""
    text = str(value)
    return text[:limit] if len(text) > limit else text

# Prefer in-memory final_df from previous cell; fallback to saved CSV.
if "final_df" in globals() and isinstance(final_df, pd.DataFrame):
    df_truncated = final_df.copy()
else:
    df_truncated = pd.read_csv(OUTPUT_CSV).fillna("")

for col in df_truncated.columns:
    df_truncated[col] = df_truncated[col].apply(truncate_value)

# Keep notebook variables aligned for downstream cells/users.
if "final_df" in globals():
    final_df = df_truncated.copy()
if "df" in globals():
    df = df_truncated.copy()

# Persist truncated output.
df_truncated.to_csv(OUTPUT_CSV, index=False)

max_len_after = int(df_truncated.astype(str).apply(lambda s: s.map(len)).max().max())
print("Truncation complete.")
print(f"Rows: {len(df_truncated)}")
print(f"Columns: {len(df_truncated.columns)}")
print(f"Max cell length after truncation: {max_len_after}")
print(f"Applied limit: {TRUNCATE_LIMIT}")

AttributeError: 'DataFrame' object has no attribute 'applymap'